# Week 2 — Task 2: Gestalt Principles of Grouping

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `stimuli_gestalt.csv` — 284 rows, 6 panels, one per Gestalt principle.

**All charts built with `fig, ax = plt.subplots()`. OO API throughout.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/stimuli_gestalt.csv')
print(f'Shape: {df.shape}')
print(f'Panels: {df["panel"].unique()}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## Step 1 — Draw All Six Panels (2×3 Figure)

**Gestalt principle relied on in layout:** Enclosure (the 2×3 grid puts each panel in its own implied boundary).

In [ ]:
panels_order = ['proximity', 'similarity', 'enclosure', 'continuity', 'closure', 'connection']
panel_colors_map = {
    'red':  '#d73027',
    'blue': '#4575b4',
    'orange': '#fc8d59',
    'green': '#1a9850'
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes_flat = axes.flatten()

for idx, panel_name in enumerate(panels_order):
    sub = df[df['panel'] == panel_name]
    ax = axes_flat[idx]
    ax.set_title(panel_name.capitalize(), fontsize=12, fontweight='bold', pad=8)
    ax.set_aspect('equal')
    ax.axis('off')

    # Determine colour mapping
    unique_colors = sub['colour'].unique()

    if panel_name == 'connection':
        # Draw connector lines first
        for connector_id in sub['connector'].dropna().unique():
            pair = sub[sub['connector'] == connector_id]
            if len(pair) == 2:
                pts = pair[['x', 'y']].values
                ax.plot([pts[0, 0], pts[1, 0]], [pts[0, 1], pts[1, 1]],
                        color='#888888', lw=1.5, zorder=1)

    for _, row in sub.iterrows():
        c = panel_colors_map.get(str(row['colour']).lower(), '#4575b4')
        s = float(row['size']) if pd.notna(row['size']) else 6
        shape = str(row['shape']).lower() if pd.notna(row['shape']) else 'circle'

        if shape == 'square':
            r = s * 0.3
            patch = mpatches.Rectangle((row['x'] - r, row['y'] - r), 2*r, 2*r,
                                       color=c, alpha=0.82, zorder=2)
        else:
            r = s * 0.3
            patch = mpatches.Circle((row['x'], row['y']), r, color=c, alpha=0.82, zorder=2)
        ax.add_patch(patch)

    if panel_name == 'enclosure':
        # Add four bounding rectangles (grey, no fill)
        for grp_id in sub['group'].unique():
            g_sub = sub[sub['group'] == grp_id]
            x0, y0 = g_sub['x'].min() - 1.2, g_sub['y'].min() - 1.2
            w = g_sub['x'].max() - g_sub['x'].min() + 2.4
            h = g_sub['y'].max() - g_sub['y'].min() + 2.4
            rect = mpatches.FancyBboxPatch((x0, y0), w, h,
                                           boxstyle='round,pad=0.3',
                                           edgecolor='#888888', facecolor='none',
                                           lw=1.5, zorder=3)
            ax.add_patch(rect)

    x_margin = (sub['x'].max() - sub['x'].min()) * 0.1 + 1
    y_margin = (sub['y'].max() - sub['y'].min()) * 0.1 + 1
    ax.set_xlim(sub['x'].min() - x_margin, sub['x'].max() + x_margin)
    ax.set_ylim(sub['y'].min() - y_margin, sub['y'].max() + y_margin)

fig.suptitle(
    'Six Gestalt grouping principles — data is nearly identical across panels;\ngrouping comes entirely from arrangement and styling',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('task2_gestalt_six_panels.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Reader Experiment: "How Many Groups Do You See?"

**Subject:** Sara (flatmate)  
**Protocol:** Each panel shown one at a time. Question: 'How many groups do you see?' No time limit.

In [ ]:
# True group counts from the dataset
true_groups = {p: df[df['panel']==p]['group'].nunique() for p in panels_order}

# Reader's responses (Sara, flatmate)
reader_groups = {
    'proximity':   4,
    'similarity':  4,
    'enclosure':   4,
    'continuity':  2,
    'closure':     2,
    'connection':  24
}

results = pd.DataFrame({
    'panel':           panels_order,
    'principle':       [p.capitalize() for p in panels_order],
    'true_groups':     [true_groups[p] for p in panels_order],
    'perceived_groups':[reader_groups[p] for p in panels_order],
    'match':           ['Yes' if true_groups[p]==reader_groups[p] else 'No'
                       for p in panels_order]
})
print(results.to_string(index=False))

## Step 3 — Principles in Conflict: Connection vs Proximity/Similarity

The `connection` panel pairs items by a connecting line; the two items in each pair are **different colours** and are **not the nearest items** to each other. We draw it twice — with and without lines — and ask the reader to group it both times.

In [ ]:
conn = df[df['panel'] == 'connection']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: WITH connecting lines (connection beats proximity and similarity)
ax = axes[0]
ax.set_title('With connecting lines\n(Principle: Connection)', fontsize=10, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

for connector_id in conn['connector'].dropna().unique():
    pair = conn[conn['connector'] == connector_id]
    if len(pair) == 2:
        pts = pair[['x', 'y']].values
        ax.plot([pts[0,0], pts[1,0]], [pts[0,1], pts[1,1]],
                color='#888888', lw=1.2, zorder=1)

for _, row in conn.iterrows():
    c = '#d73027' if str(row['colour']).lower() == 'red' else '#4575b4'
    ax.scatter(row['x'], row['y'], s=80, color=c, zorder=2, edgecolors='none')

ax.set_xlim(conn['x'].min()-1, conn['x'].max()+1)
ax.set_ylim(conn['y'].min()-1, conn['y'].max()+1)

# Panel B: WITHOUT lines (proximity and similarity dominate)
ax2 = axes[1]
ax2.set_title('Without connecting lines\n(Principle: Similarity + Proximity)', fontsize=10, fontweight='bold')
ax2.set_aspect('equal')
ax2.axis('off')

for _, row in conn.iterrows():
    c = '#d73027' if str(row['colour']).lower() == 'red' else '#4575b4'
    ax2.scatter(row['x'], row['y'], s=80, color=c, zorder=2, edgecolors='none')

ax2.set_xlim(conn['x'].min()-1, conn['x'].max()+1)
ax2.set_ylim(conn['y'].min()-1, conn['y'].max()+1)

fig.suptitle("Connection beats both proximity and similarity — the line determines grouping even across colour boundaries",
             fontsize=10)
plt.tight_layout()
plt.savefig('task2_connection_fight.png', dpi=150, bbox_inches='tight')
plt.show()

**Reader result (Sara):**
- With lines: saw 24 pairs grouped by the line — correctly identified the connection grouping even though the two ends of each line were different colours and not adjacent.
- Without lines: saw 2 groups (red items vs blue items) — similarity dominated, then proximity.

**Conclusion:** Connection beat both proximity and similarity. Once the connecting line was removed, similarity (hue) took over as the dominant grouping cue. This confirms the TASKS.md statement that **connection beats proximity and similarity**.

## Step 4 — Proximity in a Real Chart (`tips.csv`)

Grouped bar chart where within-group gap < between-group gap — proximity groups day × meal together.

**Preattentive attribute used:** Hue (lunch vs dinner).  
**Pop-out:** None deliberately — this is a comparison chart.  
**Gestalt principle:** Proximity — bars for the same day are placed closer together than bars from different days.

In [ ]:
tips = pd.read_csv('data/tips.csv')
mean_tip = tips.groupby(['day', 'time'])['tip'].mean().reset_index()
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
mean_tip['day'] = pd.Categorical(mean_tip['day'], categories=day_order, ordered=True)
mean_tip = mean_tip.sort_values('day')

x = np.arange(len(day_order))
width = 0.3

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# (A) Tight within-group gap (proximity signals grouping)
lunch = mean_tip[mean_tip['time']=='Lunch'].set_index('day').reindex(day_order)
dinner = mean_tip[mean_tip['time']=='Dinner'].set_index('day').reindex(day_order)

axes[0].bar(x - width/2, lunch['tip'].fillna(0),   width, label='Lunch',  color='#4575b4', alpha=0.85)
axes[0].bar(x + width/2, dinner['tip'].fillna(0), width, label='Dinner', color='#d73027', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(day_order)
axes[0].set_title('Proximity groups day × meal together (within-gap < between-gap)', loc='left')
axes[0].set_ylabel('Average tip ($)')
axes[0].set_ylim(0, 4.5)
axes[0].legend(frameon=False)
for sp in ('top', 'right'):
    axes[0].spines[sp].set_visible(False)

# (B) Equal gaps — proximity no longer signals grouping
equal_positions = np.arange(len(day_order) * 2) * 0.55
labels_eq = []
values_eq = []
colors_eq = []
for day in day_order:
    for time, color in [('Lunch','#4575b4'),('Dinner','#d73027')]:
        row = mean_tip[(mean_tip['day']==day) & (mean_tip['time']==time)]
        values_eq.append(row['tip'].values[0] if len(row)>0 else 0)
        labels_eq.append(f'{day}\n{time}')
        colors_eq.append(color)

axes[1].bar(equal_positions, values_eq, 0.45, color=colors_eq, alpha=0.85)
axes[1].set_xticks(equal_positions)
axes[1].set_xticklabels(labels_eq, fontsize=7)
axes[1].set_title('Equal gaps — day grouping is lost; reader must read every label', loc='left')
for sp in ('top', 'right'):
    axes[1].spines[sp].set_visible(False)

fig.suptitle('Proximity in a real chart: without the tighter within-group gap, the day grouping disappears',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('task2_proximity_tips.png', dpi=150, bbox_inches='tight')
plt.show()

**What the reader loses in the second version:** The spatial grouping that told them which bars belong to the same day is gone. They must now read every axis label individually — cognitive load rises from "scan and group" to "read, remember, decode" for each comparison. The data has not changed at all; only the spatial organisation has been neutralised.

## Step 5 — Where Each Principle Lives in a Chart

| Principle | Where it appears in real charts |
|---|---|
| **Proximity** | Grouped bar chart: bars for the same category are placed closer together than bars from different categories — the narrow gap tells the eye they belong together. |
| **Similarity** | Line chart with multiple series: each series gets a distinct hue; all segments of the same series look alike, so the eye groups them as one line even when they cross. |
| **Enclosure** | Facet / small multiples: each panel is surrounded by an implied border (axes frame or actual box), telling the reader that all marks inside belong to the same group. |
| **Closure** | Dumbbell chart or range bar: two end-points connected by a line segment suggest a closed, bounded interval — the gap in the middle is "filled in" by the brain. |
| **Continuity** | Line chart itself: segments connect along a smooth path, so the eye follows the trajectory as a single object even if individual points are spaced apart. |
| **Connection** | Connected scatter plot or slope chart: a line drawn between the same entity at two time points groups them as one observation, overriding the fact that they are plotted at different x-positions. |

## Answer: Direct Labelling vs Legend — Which Gestalt Principle?

A legend forces the reader to hold a colour in **working memory**, carry it across the chart, and match it to the data point — an extra decoding step that costs roughly one chunk of working memory and adds extraneous cognitive load.

**Direct labelling** uses **Continuity**: the label is placed at the end of the line or next to the mark it names, so the eye can follow the line directly to its annotation without leaving the chart. No memory is needed; the connection is spatial and immediate. It is cheaper for the brain because it replaces a multi-step search-and-match process with a single glance.